##In this notebook, we simulate rainfall-runoff accross 222 CAMELS-AUS catchments using the HBV model.

In [12]:
from google.colab import files

In [13]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from pandas import read_csv
import math
import random
from datetime import datetime, timedelta
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

import zipfile
import os
from scipy.optimize import minimize

##CAMELS-DATA from my Google Drive.

In [15]:
# ==============================
# Paths to ZIP files
# ==============================
hydro_zip = "/content/drive/MyDrive/Colab Notebooks/Dimension/05_hydrometeorology.zip"
streamflow_zip = "/content/drive/MyDrive/Colab Notebooks/Dimension/03_streamflow.zip"

# Data extraction directories
hydro_dir = "/content/05_hydro"
streamflow_dir = "/content/03_streamflow"

# ==============================
# Function to extract ZIP files
# ==============================
def extract_zip(zip_path, extract_to):
    if not os.path.exists(extract_to):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
        print(f"✅ ZIP extracted in {extract_to}")
    else:
        print(f"✅ Directory {extract_to} already exists")

def find_csv(base_dir, csv_name):
    # Recursive search for the CSV file
    for root, dirs, files in os.walk(base_dir):
        if csv_name in files:
            return os.path.join(root, csv_name)
    raise FileNotFoundError(f"{csv_name} not found in {base_dir}")

# ==============================
# Extraction
# ==============================
extract_zip(hydro_zip, hydro_dir)
extract_zip(streamflow_zip, streamflow_dir)

# ==============================
# Load the 222 basins data
# ==============================
file_path = '/content/drive/MyDrive/Colab Notebooks/Dimension/id_name_metadata.csv'
basin222 = pd.read_csv(file_path)
station_ids_v1 = basin222['station_id'].astype(str).str.strip().unique()
print(f"✅ {len(station_ids_v1)} official basins loaded")

# ==============================
# 1️⃣ Precipitation (SILO)
# ==============================
precip_file = find_csv(hydro_dir, "precipitation_SILO.csv")
precip = pd.read_csv(precip_file, index_col=0, parse_dates=True)
precip.columns = precip.columns.str.strip()
precip.replace(-99.99, np.nan, inplace=True)
print("✅ SILO precipitation:", precip.shape)

# ==============================
# 2️⃣ Evapotranspiration (ET SILO)
# ==============================
et_file = find_csv(hydro_dir, "et_morton_actual_SILO.csv")
et = pd.read_csv(et_file, index_col=0, parse_dates=True)
et.columns = et.columns.str.strip()
et.replace(-99.99, np.nan, inplace=True)
print("✅ SILO ET:", et.shape)

# ==============================
# 3️⃣ Streamflow
# ==============================
streamflow_file = find_csv(streamflow_dir, "streamflow_mmd.csv")
Q = pd.read_csv(streamflow_file, index_col=0, parse_dates=True)
Q.columns = Q.columns.str.strip()
Q.replace(-99.99, np.nan, inplace=True)
print("✅ Streamflow:", Q.shape)

# ==============================
# 4️⃣ Identify common stations
# ==============================
stations_precip = set(precip.columns)
stations_et = set(et.columns)
stations_Q = set(Q.columns)

common_stations = [
    s for s in station_ids_v1
    if s in stations_precip and s in stations_et and s in stations_Q
]

print(f"✅ Official common stations: {len(common_stations)}")

# ==============================
# 5️⃣ Subset common stations
# ==============================
precip = precip[common_stations]
et = et[common_stations]
Q = Q[common_stations]

# ==============================
# 6️⃣ Final verification
# ==============================
print("Precipitation:", precip.shape)
print("ET:", et.shape)
print("Streamflow:", Q.shape)
print("Stations (first 10):", common_stations[:10], "...")


✅ Directory /content/05_hydro already exists
✅ Directory /content/03_streamflow already exists
✅ 222 official basins loaded
✅ SILO precipitation: (43464, 224)
✅ SILO ET: (43464, 224)
✅ Streamflow: (23376, 224)
✅ Official common stations: 222
Precipitation: (43464, 222)
ET: (43464, 222)
Streamflow: (23376, 222)
Stations (first 10): ['912101A', '912105A', '915011A', '917107A', '919003A', '919201A', '919309A', '922101B', '925001A', '926002A'] ...


GR4J

In [16]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from numba import njit

# ============================================
# 1️⃣ General setting
# ============================================
start_date = "1980-01-01"
end_date = "2014-12-31"
b1_ratio = 0.7
max_missing_ratio = 1.0
stations = common_stations
results_HBV = {}

# ============================================
# 2️⃣ Metrics with Numba
# ============================================
@njit
def NSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0 or np.var(obs) == 0:
        return np.nan
    return 1 - np.sum((sim - obs)**2) / np.sum((obs - np.mean(obs))**2)

@njit
def NNSE(obs, sim):
    nse = NSE(obs, sim)
    if np.isnan(nse):
        return np.nan
    return 1 / (2 - nse)

@njit
def RMSE(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0:
        return np.nan
    return np.sqrt(np.mean((sim - obs)**2))

@njit
def PBIAS(obs, sim):
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0 or np.sum(obs) == 0:
        return np.nan
    return  np.sum(sim - obs) / np.sum(obs)

@njit
def FHV(obs, sim, top_fraction=0.02):
    epsilon = 0
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0:
        return np.nan
    n_top = int(len(obs) * top_fraction)
    if n_top == 0:
        return np.nan
    idx = np.argsort(obs)[-n_top:]
    obs_top = obs[idx]
    sim_top = sim[idx]
    return np.sum(sim_top - obs_top) / (np.sum(obs_top) + epsilon)

@njit
def FLV(obs, sim, bottom_fraction=0.3):
    epsilon = 1e-6
    mask = ~np.isnan(obs) & ~np.isnan(sim)
    obs = obs[mask]
    sim = sim[mask]
    if len(obs) == 0:
        return np.nan
    n_bot = int(len(obs) * bottom_fraction)
    if n_bot == 0:
        return np.nan
    idx = np.argsort(obs)[:n_bot]
    obs_bot = obs[idx]
    sim_bot = sim[idx]
    return  np.sum(sim_bot - obs_bot) / (np.sum(obs_bot) + epsilon)

# ============================================
# 3️⃣ HBV
# ============================================
@njit
def hbv_numba(P, ET, params):
    FC, BETA, LP, K0, K1, K2, UZL, PERC = params

    SM = 0.5 * FC
    UZ = 0.0
    LZ = 0.0

    n = len(P)
    Qsim = np.zeros(n)

    for t in range(n):
        p = P[t]
        et = ET[t]

        # Soil moisture
        soil_ratio = SM / FC
        recharge = p * soil_ratio**BETA
        SM += p - recharge
        if SM > FC:
            recharge += SM - FC
            SM = FC
        evap = min(et * min(SM / (LP * FC), 1.0), SM)
        SM -= evap

        # Upper reservoir
        UZ += recharge
        perc = min(PERC, UZ)
        UZ -= perc
        LZ += perc

        # Quick flow
        Q0 = K0 * (UZ - UZL) if UZ > UZL else 0.0
        if Q0 > 0:
            UZ -= Q0

        # Interflow
        Q1 = K1 * UZ
        UZ -= Q1

        # Baseflow
        Q2 = K2 * LZ
        LZ -= Q2

        # Total discharge
        Qsim[t] = max(0.0, Q0 + Q1 + Q2)

    return Qsim

# ============================================
# 4️⃣ Fonction objective
# ============================================
def objective_hbv(x, Q, P, ET):
    Qsim = hbv_numba(P, ET, x)
    nse = NSE(Q, Qsim)
    return 1 - nse if np.isfinite(nse) else 1e6

# ============================================
# 5️⃣ Boucle sur stations
# ============================================
bounds = [
    (50,1000), (1,6), (0.3,1), (0,0.5), (0,0.3),
    (0,0.1), (0,100), (0,5)
]

for i, station_id in enumerate(stations, start=1):
    print(f"\n=== Station {station_id} ({i}/{len(stations)}) ===")

    Q_obs = Q[station_id].loc[start_date:end_date].to_numpy(float)
    P = precip[station_id].loc[start_date:end_date].to_numpy(float)
    ET = et[station_id].loc[start_date:end_date].to_numpy(float)

    N = len(Q_obs)
    if N == 0 or np.all(np.isnan(Q_obs)):
        continue

    missing_ratio = np.sum(np.isnan(Q_obs)) / N
    if missing_ratio > max_missing_ratio:
        continue

    # Calibration / validation
    b1 = int(N * b1_ratio)
    Q_cal, Q_val = Q_obs[:b1], Q_obs[b1:]
    P_cal, P_val = P[:b1], P[b1:]
    ET_cal, ET_val = ET[:b1], ET[b1:]

    # Multi-start calibration
    best_fun = np.inf
    best_x = None
    np.random.seed(42)
    for _ in range(10):
        x0 = [np.random.uniform(b[0], b[1]) for b in bounds]
        res = minimize(
            objective_hbv,
            x0,
            args=(Q_cal, P_cal, ET_cal),
            method="L-BFGS-B",
            bounds=bounds,
            options={"maxiter":3000}
        )
        if res.fun < best_fun:
            best_fun = res.fun
            best_x = res.x

    if best_x is None:
        continue

    Qsim_cal = hbv_numba(P_cal, ET_cal, best_x)
    Qsim_val = hbv_numba(P_val, ET_val, best_x)

    # Metrics
    NSE_cal = NSE(Q_cal, Qsim_cal)
    NNSE_cal = NNSE(Q_cal, Qsim_cal)
    RMSE_cal = RMSE(Q_cal, Qsim_cal)
    PBIAS_cal = PBIAS(Q_cal, Qsim_cal)
    FHV_cal = FHV(Q_cal, Qsim_cal)
    FLV_cal = FLV(Q_cal, Qsim_cal)

    NSE_val = NSE(Q_val, Qsim_val)
    NNSE_val = NNSE(Q_val, Qsim_val)
    RMSE_val = RMSE(Q_val, Qsim_val)
    PBIAS_val = PBIAS(Q_val, Qsim_val)
    FHV_val = FHV(Q_val, Qsim_val)
    FLV_val = FLV(Q_val, Qsim_val)

    # Store results
    results_HBV[station_id] = {
        "NSE_cal": NSE_cal,
        "NNSE_cal": NNSE_cal,
        "RMSE_cal": RMSE_cal,
        "PBIAS_cal": PBIAS_cal,
        "FHV_cal": FHV_cal,
        "FLV_cal": FLV_cal,
        "NSE_val": NSE_val,
        "NNSE_val": NNSE_val,
        "RMSE_val": RMSE_val,
        "PBIAS_val": PBIAS_val,
        "FHV_val": FHV_val,
        "FLV_val": FLV_val,
        **{k:v for k,v in zip(["FC","BETA","LP","K0","K1","K2","UZL","PERC"], best_x)},
        "missing_ratio": missing_ratio
    }

    # Display
    print(f"NSE_cal: {NSE_cal:.3f}, NSE_val: {NSE_val:.3f}")
    print(f"Params: {best_x}")

print(f"\n✅ Terminé : {len(results_HBV)} bassins calibrés")


=== Station 912101A (1/222) ===
NSE_cal: 0.524, NSE_val: 0.572
Params: [1.65981483e+02 2.66785295e+00 9.10551060e-01 7.27429850e-02
 7.45535876e-02 3.28324852e-05 3.12191121e+01 5.00000000e+00]

=== Station 912105A (2/222) ===
NSE_cal: 0.680, NSE_val: 0.564
Params: [1.57142514e+02 1.00000000e+00 9.85061785e-01 1.53797812e-01
 7.07033591e-02 1.63650570e-05 6.33302501e+01 4.99999999e+00]

=== Station 915011A (3/222) ===
NSE_cal: 0.598, NSE_val: 0.675
Params: [6.16605841e+02 1.95456204e+00 1.00000000e+00 5.00000000e-01
 1.96948888e-01 9.02726813e-06 4.60454650e+01 5.00000000e+00]

=== Station 917107A (4/222) ===
NSE_cal: 0.756, NSE_val: 0.612
Params: [6.31367465e+02 1.00000000e+00 6.36659380e-01 2.26251049e-01
 1.40766142e-01 2.01180294e-05 2.06532933e+01 5.00000000e+00]

=== Station 919003A (5/222) ===
NSE_cal: 0.648, NSE_val: 0.728
Params: [2.77033180e+02 2.36657491e+00 7.47481995e-01 5.14286788e-02
 1.87407464e-01 8.48014391e-05 3.40181039e+01 8.17077992e-01]

=== Station 919201A (6/2

In [17]:
# =============================================================
# 📌 FUNCTION TO PRINT METRIC SUMMARY
# =============================================================
def summarize_metric(metric_name, results_dict):

    values = [res.get(metric_name, np.nan) for res in results_dict.values()]
    values = np.array(values)
    values = values[~np.isnan(values)]  # remove NaN

    if len(values) == 0:
        print(f"{metric_name}: No valid values\n")
        return

    print(f"{metric_name}")
    print(f"Median : {np.percentile(values,50):.3f}")
    print(f"Mean   : {np.mean(values):.3f}")
    print(f"Min    : {np.min(values):.3f}")
    print(f"Max    : {np.max(values):.3f}")
    print(f"5th–95th percentile : {np.percentile(values,5):.3f} – {np.percentile(values,95):.3f}")
    print("-"*40)


# =============================================================
# 📌 CALIBRATION SUMMARY
# =============================================================
print("\n================= CALIBRATION SUMMARY =================\n")

metrics_cal = [
    "NSE_cal",
    "NNSE_cal",
    "RMSE_cal",
    "PBIAS_cal",
    "FHV_cal",
    "FLV_cal"
]

for metric in metrics_cal:
    summarize_metric(metric, results_HBV)


# =============================================================
# 📌 VALIDATION SUMMARY
# =============================================================
print("\n================= VALIDATION SUMMARY =================\n")

metrics_val = [
    "NSE_val",
    "NNSE_val",
    "RMSE_val",
    "PBIAS_val",
    "FHV_val",
    "FLV_val"
]

for metric in metrics_val:
    summarize_metric(metric, results_HBV)


================= CALIBRATION SUMMARY =================

NSE_cal
Median : 0.732
Mean   : 0.710
Min    : 0.357
Max    : 0.900
5th–95th percentile : 0.501 – 0.862
----------------------------------------
NNSE_cal
Median : 0.788
Mean   : 0.781
Min    : 0.609
Max    : 0.909
5th–95th percentile : 0.667 – 0.879
----------------------------------------
RMSE_cal
Median : 0.666
Mean   : 1.147
Min    : 0.027
Max    : 6.910
5th–95th percentile : 0.125 – 3.284
----------------------------------------
PBIAS_cal
Median : -0.024
Mean   : -0.005
Min    : -0.203
Max    : 0.400
5th–95th percentile : -0.123 – 0.212
----------------------------------------
FHV_cal
Median : -0.236
Mean   : -0.240
Min    : -0.582
Max    : -0.024
5th–95th percentile : -0.432 – -0.084
----------------------------------------
FLV_cal
Median : 2.696
Mean   : 12837952.534
Min    : -0.508
Max    : 433306872.211
5th–95th percentile : -0.039 – 67376390.930
----------------------------------------

================= VALIDATION SUMM

# Save

In [18]:
# ============================================
# Save in excel file
# ============================================

records = []
for station_id, res in results_HBV.items():
    record = {
        "Station": station_id,
        "NSE_val": res["NSE_val"],
        "NNSE_val": res["NNSE_val"],
        "RMSE_val": res["RMSE_val"],
        "PBIAS_val": res["PBIAS_val"],
        "FHV_val": res["FHV_val"],
        "FLV_val": res["FLV_val"],
    }
    records.append(record)

df_val_metrics = pd.DataFrame(records)
excel_filename = "HBV_validation_metrics.xlsx"
df_val_metrics.to_excel(excel_filename, index=False)
print(f"✅ Excel saved: {excel_filename}")

#
try:
    from google.colab import files
    #files.download(excel_filename)
except:
    print("⚠️ Not running in Colab. File saved locally.")

✅ Excel saved: HBV_validation_metrics.xlsx
